**Dataset**
labeled datasset collected from Spotify (Assignment 1 - Spotify Reviews Rating)

**Objective**
classify Review to a category from 1 to 5. <br>

**Total Estimated Time = 90-120 Mins**

**Evaluation metric**
macro f1 score

### Import used libraries

In [1]:
import pandas as pd

### Load Dataset

In [2]:
spotify_data = pd.read_csv("data/Lab 2 - Spotify Reviews Rating.csv")
spotify_data.head()

,Time_submitted,Review,Rating
0,7/9/2022 15:00,"Great music service, the audio is high quality...",5
1,7/9/2022 14:21,Please ignore previous negative rating. This a...,5
2,7/9/2022 13:27,"This pop-up ""Get the best Spotify experience o...",4
3,7/9/2022 13:26,Really buggy and terrible to use as of recently,1
4,7/9/2022 13:20,Dear Spotify why do I get songs that I didn't ...,1


In [3]:
spotify_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61594 entries, 0 to 61593
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Time_submitted  61594 non-null  object
 1   Review          61594 non-null  object
 2   Rating          61594 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.4+ MB


### Data splitting

It is a good practice to split the data before EDA helps maintain the integrity of the machine learning process, prevents data leakage, simulates real-world scenarios more accurately, and ensures reliable model performance evaluation on unseen data.

In [4]:
from sklearn.model_selection import train_test_split
X = spotify_data["Review"]
y = spotify_data["Rating"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train_size: {len(x_train)}\nTest_size: {len(x_test)}")

train_size: 49275
Test_size: 12319


### EDA on training data

- check NaNs

In [5]:
spotify_data.isnull().sum()

Time_submitted    0
Review            0
Rating            0
dtype: int64

- check duplicates

In [6]:
spotify_data.duplicated().sum()

np.int64(0)

- show a representative sample of data texts to find out required preprocessing steps

In [7]:
import random
random_indices = random.sample(range(len(spotify_data)), 5)
for index in random_indices:
	print(f"index: {index}\nReview: {spotify_data.loc[index, 'Review']}\nRating: {spotify_data.loc[index, 'Rating']}\n")

index: 12279
Review: Awesome music player but I just went to listen to my Liked Playlist and listened to 8 ads in a row. I think it could be sooooo much better if they changed to 2 ads or maybe just three every now and then. Other than that they have most of the songs I like.
Rating: 3

index: 45615
Review: Super steller dude. It's my one and only music streaming platform.
Rating: 5

index: 40720
Review: Too much advertisement, after every song. I am fedup with this, I am uninstalling this Application.
Rating: 1

index: 5109
Review: Spotify is the best app
Rating: 5

index: 50466
Review: The offline playback is garbage for downloaded stuff. I would go back to iTunes if it wasn't for my library and Rogan.
Rating: 2



- check dataset balancing

In [8]:
normalized_data=spotify_data["Rating"].value_counts(normalize=True)
count_data = spotify_data["Rating"].value_counts()
print(f"count data: {count_data}\nnormalized data: {normalized_data}")

count data: Rating
5    22095
1    17653
4     7842
2     7118
3     6886
Name: count, dtype: int64
normalized data: Rating
5    0.358720
1    0.286603
4    0.127318
2    0.115563
3    0.111797
Name: proportion, dtype: float64


* data not balanced and not unbalanced much but class 5 which means they like app is more than rest percentage wise

- Cleaning and Preprocessing are:
    - 1 lower case the data
    - 2 fix contractions "i'll" to be "i will"
    - 3 remove punctuations
    - 4 tokenize text
	- 5 remove stopwords
	- 6 lemmitization

### Cleaning and Preprocessing

In [9]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
import string
from sklearn.base import BaseEstimator, TransformerMixin

class clean_preprocess_text(BaseEstimator, TransformerMixin):
	def __init__(self, target_col=None):
		self.target_col = target_col
		self.stop_words = set(stopwords.words("english"))
		self.lemmatizer = WordNetLemmatizer()

	def clean_text(self, text):
		text = str(text).lower()
		text = re.sub(r'[^\x00-\x7F]+', '', text)
		text = contractions.fix(text)
		text = text.translate(str.maketrans("", "", string.punctuation))
		tokens = word_tokenize(text)
		tokens = [word for word in tokens if word not in self.stop_words]
		og_tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
		return " ".join(og_tokens)

	def fit(self, X, y=None):
		return self

	def transform(self, X):
		X_transformed = X.copy()
		if isinstance(X, pd.DataFrame) and self.target_col in X.columns:
			X_transformed[self.target_col] = X_transformed[self.target_col].apply(self.clean_text)
		else:
			X_transformed = X_transformed.apply(self.clean_text) ## the data is series anyway but best practice just incase
		return X_transformed
        

In [10]:
test_one_row = spotify_data["Review"].iloc[0]
cleaner = clean_preprocess_text()
cleaned_review = cleaner.clean_text(test_one_row)
print(f"Original review: {test_one_row}\nCleaned review: {cleaned_review}")

Original review: Great music service, the audio is high quality and the app is easy to use. Also very quick and friendly support.
Cleaned review: great music service audio high quality app easy use also quick friendly support


**You  are doing Great so far!**

### Modelling

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
CV = CountVectorizer()
TFIDF = TfidfVectorizer()
model = LogisticRegression(max_iter=1000, solver="saga")

In [12]:
x_transf = cleaner.fit_transform(x_train)
print(f"og_text: {x_train.iloc[0]}\ntransformed_text: {x_transf.iloc[0]}")

og_text: Making it different from other's is Podcast.... Love the app
transformed_text: making different others podcast love app


In [13]:
pipeline_cv = Pipeline([
	("cleaner", clean_preprocess_text()),
	("vectorizer", CV),
	("model", model)
])
pipeline_ifidf = Pipeline([
	("cleaner", clean_preprocess_text()),
	("vectorizer", TFIDF),
	("model", model)
])

In [14]:
pipeline_cv.fit(x_train, y_train)
pipeline_ifidf.fit(x_train, y_train)
print(f"acc of cv: {pipeline_cv.score(x_test, y_test)}\nacc of tfidf: {pipeline_ifidf.score(x_test, y_test)}")

acc of cv: 0.5616527315528858
acc of tfidf: 0.6162837892686095


#### Evaluation

**Evaluation metric:**
macro f1 score

Macro F1 score is a useful metric in scenarios where you want to evaluate the overall performance of a multi-class classification model, **particularly when the classes are imbalanced**

![Calculation](https://assets-global.website-files.com/5d7b77b063a9066d83e1209c/639c3d934e82c1195cdf3c60_macro-f1.webp)

In [15]:
from sklearn.metrics import classification_report

In [16]:
y_pred_cv = pipeline_cv.predict(x_test)
y_pred_tfidf = pipeline_ifidf.predict(x_test)
class_rep_cv = classification_report(y_test, y_pred_cv)
class_rep_tfidf = classification_report(y_test, y_pred_tfidf)
print(f"Classification report for CountVectorizer:\n{class_rep_cv}")
print(f"Classification report for TfidfVectorizer:\n{class_rep_tfidf}")

Classification report for CountVectorizer:
              precision    recall  f1-score   support

           1       0.66      0.58      0.62      3531
           2       0.25      0.41      0.31      1424
           3       0.26      0.21      0.23      1377
           4       0.36      0.42      0.39      1568
           5       0.85      0.75      0.80      4419

    accuracy                           0.56     12319
   macro avg       0.47      0.47      0.47     12319
weighted avg       0.60      0.56      0.57     12319

Classification report for TfidfVectorizer:
              precision    recall  f1-score   support

           1       0.58      0.85      0.69      3531
           2       0.31      0.10      0.15      1424
           3       0.31      0.13      0.18      1377
           4       0.42      0.26      0.32      1568
           5       0.75      0.87      0.81      4419

    accuracy                           0.62     12319
   macro avg       0.47      0.44      0.43  

* best macro f1 avg is f1 score

### Enhancement

In [17]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "vectorizer__ngram_range": [(1, 3)],
    "vectorizer__max_df": [0.85, 0.9],
    "vectorizer__min_df": [2, 5],
    "vectorizer__max_features": [None, 5000]
}

grid_search_tfidf = GridSearchCV(estimator=pipeline_ifidf, param_grid=param_grid, cv=5)
grid_search_tfidf.fit(x_train, y_train)
print(f"Best parameters: {grid_search_tfidf.best_params_}")
best_tfidf_model = grid_search_tfidf.best_estimator_

Best parameters: {'vectorizer__max_df': 0.85, 'vectorizer__max_features': None, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 3)}


In [18]:
y_pred_tfidf_best = best_tfidf_model.predict(x_test)
class_rep_tfidf_best = classification_report(y_test, y_pred_tfidf_best)
print(f"best TfidfVectorizer model:\n{class_rep_tfidf_best}")

best TfidfVectorizer model:
              precision    recall  f1-score   support

           1       0.57      0.86      0.69      3531
           2       0.27      0.09      0.13      1424
           3       0.31      0.11      0.16      1377
           4       0.42      0.25      0.31      1568
           5       0.76      0.88      0.81      4419

    accuracy                           0.62     12319
   macro avg       0.47      0.44      0.42     12319
weighted avg       0.56      0.62      0.56     12319



### Conclusion and final results


In [19]:
print(f"best TfidfVectorizer model:\n{class_rep_tfidf_best}")

best TfidfVectorizer model:
              precision    recall  f1-score   support

           1       0.57      0.86      0.69      3531
           2       0.27      0.09      0.13      1424
           3       0.31      0.11      0.16      1377
           4       0.42      0.25      0.31      1568
           5       0.76      0.88      0.81      4419

    accuracy                           0.62     12319
   macro avg       0.47      0.44      0.42     12319
weighted avg       0.56      0.62      0.56     12319



* when i tried normal params of 2 vectorizers (CV and TFIDF)
	- TFIDF performed better
	- on params i got best params from small gridsearch

#### Done!

* this part for the Word embedding in a pipeline on its own

In [34]:
from gensim.models import Word2Vec
import numpy as np

class word2vec_vect(BaseEstimator, TransformerMixin):
    def __init__(self, targ_col=None, vector_size=100, window=5, min_count=1):
        self.targ_col = targ_col
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.model = None
    
    def fit(self, X, y=None):
        self.model = Word2Vec(
			 	sentences=X,
            	vector_size=self.vector_size,
                window=self.window,
                min_count=self.min_count       
		)
        return self
    
    def transform(self, X):
        X_transformed = np.zeros((len(X), self.vector_size))
        
        for i, text in enumerate(X):
            valid_words = [self.model.wv[word] for word in text if word in self.model.wv]
            if valid_words:
                X_transformed[i] = np.mean(valid_words, axis=0)
        return X_transformed

In [35]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
import string
from sklearn.base import BaseEstimator, TransformerMixin

class word2vec_clean_preprocess_text(BaseEstimator, TransformerMixin):
	def __init__(self, target_col=None):
		self.target_col = target_col
		self.stop_words = set(stopwords.words("english"))
		self.lemmatizer = WordNetLemmatizer()

	def clean_text(self, text):
		text = str(text).lower()
		text = re.sub(r'[^\x00-\x7F]+', '', text)
		text = contractions.fix(text)
		text = text.translate(str.maketrans("", "", string.punctuation))
		tokens = word_tokenize(text)
		tokens = [word for word in tokens if word not in self.stop_words]
		og_tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
		return og_tokens

	def fit(self, X, y=None):
		return self

	def transform(self, X):
		X_transformed = X.copy()
		if isinstance(X, pd.DataFrame) and self.target_col in X.columns:
			X_transformed[self.target_col] = X_transformed[self.target_col].apply(self.clean_text)
		else:
			X_transformed = X_transformed.apply(self.clean_text) ## the data is series anyway but best practice just incase
		return X_transformed
        

In [36]:

word2vec_pipeline = Pipeline([
	("cleaner", word2vec_clean_preprocess_text()),
	("vectorizer", word2vec_vect(targ_col="review")),
	("model", model)
])
word2vec_pipeline.fit(x_train, y_train)
y_pred_word2vec = word2vec_pipeline.predict(x_test)

In [37]:
class_rep_word2vec = classification_report(y_test, y_pred_word2vec, zero_division=0)
print(f"Classification report for Word2Vec:\n{class_rep_word2vec}")

Classification report for Word2Vec:
              precision    recall  f1-score   support

           1       0.52      0.88      0.66      3531
           2       0.22      0.02      0.03      1424
           3       0.25      0.04      0.08      1377
           4       0.41      0.19      0.26      1568
           5       0.72      0.86      0.79      4419

    accuracy                           0.59     12319
   macro avg       0.42      0.40      0.36     12319
weighted avg       0.51      0.59      0.52     12319



* i expected word2vec would get better macro f1 score than TFIDF but somehow it didnt happen, maybe if i changed the hyperparameters i would get a better metric

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_w2v = {
    'vectorizer__vector_size': [100, 200, 300],
    'vectorizer__window': [3, 5],
    'vectorizer__min_count': [1, 2],
}
grid_search_w2v = GridSearchCV(estimator=word2vec_pipeline, param_grid=param_grid_w2v, cv=5)
grid_search_w2v.fit(x_train, y_train)
print(f"Best parameters: {grid_search_w2v.best_params_}")
best_w2v_model = grid_search_w2v.best_estimator_
y_pred_word2vec_best = best_w2v_model.predict(x_test)
class_rep_word2vec_best = classification_report(y_test, y_pred_word2vec_best, zero_division=0)
print(f"best Word2Vec model:\n{class_rep_word2vec_best}")